# KYC Thesis — Model 3: Liveness Detection (Inference-First, Thesis Rigor)

This notebook adds:
- FP32 vs INT8 vs Distilled comparison
- Multi-seed runs (mean ± std)
- Bootstrap 95% CI
- McNemar significance tests
- Drive-persisted progress, metrics, plots, ONNX exports


In [ ]:
%%capture
!pip install -q timm==1.0.8 scikit-learn==1.5.1 statsmodels==0.14.2 onnx==1.16.2 onnxruntime==1.18.1


In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

from sklearn.metrics import confusion_matrix, roc_curve, auc
from statsmodels.stats.contingency_tables import mcnemar

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/kyc_thesis')
DATA_ROOT = BASE_DIR / 'data' / 'liveness' / 'celeba_spoof'
RUN_ID = f'liveness_rigorous_{int(time.time())}'
RUN_DIR = BASE_DIR / 'experiments' / 'liveness' / RUN_ID
REPORT_DIR = RUN_DIR / 'reports'
EXPORT_DIR = RUN_DIR / 'exports'
CKPT_DIR = RUN_DIR / 'checkpoints'
MATRIX_CSV = BASE_DIR / 'reports' / 'comparison_matrix.csv'

for d in [RUN_DIR, REPORT_DIR, EXPORT_DIR, CKPT_DIR, BASE_DIR / 'reports']:
    d.mkdir(parents=True, exist_ok=True)

PROGRESS_JSON = RUN_DIR / 'progress.json'

def save_progress(stage, status='done', **extra):
    payload = {'run_id': RUN_ID, 'stage': stage, 'status': status, 'time': int(time.time())}
    payload.update(extra)
    PROGRESS_JSON.write_text(json.dumps(payload, indent=2))

save_progress('init')
print('RUN_DIR:', RUN_DIR)


In [ ]:
# Inference-first defaults
RUN_TRAIN_FP32 = False
RUN_TRAIN_DISTILL = False

SEEDS = [42, 43, 44]
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS_FP32 = 8
EPOCHS_DISTILL = 6
LR = 1e-4

# If not training, provide existing checkpoints under these names for each seed.
# Example: liveness_fp32_seed42.pt, liveness_distilled_seed42.pt

for split in ['train', 'val', 'test']:
    assert (DATA_ROOT / split).exists(), f'Missing {DATA_ROOT / split}'

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_tf)
val_ds = datasets.ImageFolder(DATA_ROOT / 'val', transform=eval_tf)
test_ds = datasets.ImageFolder(DATA_ROOT / 'test', transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Class mapping:', train_ds.class_to_idx)
save_progress('data_loaded', train=len(train_ds), val=len(val_ds), test=len(test_ds))


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def build_fp32_model():
    return timm.create_model('mobilenetv2_100', pretrained=True, num_classes=1).to(DEVICE)

def build_student_model():
    return timm.create_model('mobilenetv3_small_050', pretrained=True, num_classes=1).to(DEVICE)

def eval_split(model, loader):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE).float()
            logits = model(x).squeeze(1)
            prob = torch.sigmoid(logits)
            y_true.extend(y.cpu().numpy().tolist())
            y_prob.extend(prob.cpu().numpy().tolist())
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = float(auc(fpr, tpr))
    acc = float((y_true == y_pred).mean())

    cm = confusion_matrix(y_true, y_pred, labels=[1,0])
    tp, fn = cm[0,0], cm[0,1]
    fp, tn = cm[1,0], cm[1,1]
    apcer = fp / max(fp + tn, 1)
    bpcer = fn / max(tp + fn, 1)
    acer = 0.5 * (apcer + bpcer)

    return {
      'acc': acc, 'auc': roc_auc, 'apcer': float(apcer), 'bpcer': float(bpcer), 'acer': float(acer),
      'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob
    }

def train_fp32(seed):
    set_seed(seed)
    model = build_fp32_model()
    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    best_acer = 999.0
    path = CKPT_DIR / f'liveness_fp32_seed{seed}.pt'

    for epoch in range(1, EPOCHS_FP32 + 1):
        model.train()
        for x, y in train_loader:
            x = x.to(DEVICE); y = y.to(DEVICE).float()
            opt.zero_grad(set_to_none=True)
            logits = model(x).squeeze(1)
            loss = crit(logits, y)
            loss.backward()
            opt.step()

        valm = eval_split(model, val_loader)
        if valm['acer'] < best_acer:
            best_acer = valm['acer']
            torch.save(model.state_dict(), path)

    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model, path

def train_distilled(seed, teacher):
    set_seed(seed)
    student = build_student_model()
    ce = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()
    opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=1e-4)
    best_acer = 999.0
    path = CKPT_DIR / f'liveness_distilled_seed{seed}.pt'

    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    for epoch in range(1, EPOCHS_DISTILL + 1):
        student.train()
        for x, y in train_loader:
            x = x.to(DEVICE); y = y.to(DEVICE).float()
            opt.zero_grad(set_to_none=True)
            s = student(x).squeeze(1)
            with torch.no_grad():
                t = teacher(x).squeeze(1)
            loss = 0.6 * ce(s, y) + 0.4 * mse(s, t)
            loss.backward()
            opt.step()

        valm = eval_split(student, val_loader)
        if valm['acer'] < best_acer:
            best_acer = valm['acer']
            torch.save(student.state_dict(), path)

    student.load_state_dict(torch.load(path, map_location=DEVICE))
    return student, path

def latency_ms(model, runs=100):
    model = model.cpu().eval()
    x = torch.randn(1,3,IMG_SIZE,IMG_SIZE)
    with torch.no_grad():
        for _ in range(10):
            _ = model(x)
    ts = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = model(x)
            ts.append((time.perf_counter()-t0)*1000)
    return float(np.mean(ts))

def size_mb(model):
    p = CKPT_DIR / 'tmp_size.pth'
    torch.save(model.state_dict(), p)
    s = p.stat().st_size / 1e6
    p.unlink(missing_ok=True)
    return float(s)

def bootstrap_ci(values, n=2000, alpha=0.05):
    vals = np.array(values)
    boots = []
    for _ in range(n):
        sample = np.random.choice(vals, size=len(vals), replace=True)
        boots.append(sample.mean())
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi


In [ ]:
rows = []
all_preds = {'fp32': [], 'int8': [], 'distill': []}
all_true = []

for seed in SEEDS:
    set_seed(seed)

    fp32_path = CKPT_DIR / f'liveness_fp32_seed{seed}.pt'
    if RUN_TRAIN_FP32 or (not fp32_path.exists()):
        fp32_model, fp32_path = train_fp32(seed)
    else:
        fp32_model = build_fp32_model()
        fp32_model.load_state_dict(torch.load(fp32_path, map_location=DEVICE))

    fp32_m = eval_split(fp32_model, test_loader)
    fp32_lat = latency_ms(fp32_model)
    fp32_sz = size_mb(fp32_model)

    q_model = torch.quantization.quantize_dynamic(fp32_model.cpu(), {nn.Linear}, dtype=torch.qint8)
    q_model = q_model.to(DEVICE)
    int8_m = eval_split(q_model, test_loader)
    int8_lat = latency_ms(q_model)
    int8_sz = size_mb(q_model)

    dist_path = CKPT_DIR / f'liveness_distilled_seed{seed}.pt'
    if RUN_TRAIN_DISTILL or dist_path.exists():
        if RUN_TRAIN_DISTILL and (not dist_path.exists()):
            dist_model, dist_path = train_distilled(seed, fp32_model)
        else:
            dist_model = build_student_model()
            dist_model.load_state_dict(torch.load(dist_path, map_location=DEVICE))

        dist_m = eval_split(dist_model, test_loader)
        dist_lat = latency_ms(dist_model)
        dist_sz = size_mb(dist_model)
    else:
        dist_m = None

    rows.append({'seed': seed, 'variant': 'fp32', 'acc': fp32_m['acc'], 'auc': fp32_m['auc'], 'apcer': fp32_m['apcer'], 'bpcer': fp32_m['bpcer'], 'acer': fp32_m['acer'], 'latency_ms': fp32_lat, 'size_mb': fp32_sz})
    rows.append({'seed': seed, 'variant': 'int8', 'acc': int8_m['acc'], 'auc': int8_m['auc'], 'apcer': int8_m['apcer'], 'bpcer': int8_m['bpcer'], 'acer': int8_m['acer'], 'latency_ms': int8_lat, 'size_mb': int8_sz})
    if dist_m is not None:
        rows.append({'seed': seed, 'variant': 'distilled', 'acc': dist_m['acc'], 'auc': dist_m['auc'], 'apcer': dist_m['apcer'], 'bpcer': dist_m['bpcer'], 'acer': dist_m['acer'], 'latency_ms': dist_lat, 'size_mb': dist_sz})

    all_true.append(fp32_m['y_true'])
    all_preds['fp32'].append(fp32_m['y_pred'])
    all_preds['int8'].append(int8_m['y_pred'])
    if dist_m is not None:
        all_preds['distill'].append(dist_m['y_pred'])

    save_progress('seed_done', seed=seed)

rdf = pd.DataFrame(rows)
rdf.to_csv(REPORT_DIR / 'per_seed_metrics.csv', index=False)
save_progress('per_seed_saved')
rdf.head()


In [ ]:
summary = rdf.groupby('variant').agg(['mean','std'])
summary.columns = ['_'.join(c) for c in summary.columns]
summary = summary.reset_index()

ci_rows = []
for v in rdf['variant'].unique():
    vv = rdf[rdf['variant']==v]
    ci_acc = bootstrap_ci(vv['acc'].values)
    ci_auc = bootstrap_ci(vv['auc'].values)
    ci_acer = bootstrap_ci(vv['acer'].values)
    ci_rows.append({'variant': v, 'acc_ci95': ci_acc, 'auc_ci95': ci_auc, 'acer_ci95': ci_acer})

ci_df = pd.DataFrame(ci_rows)
summary.to_csv(REPORT_DIR / 'summary_mean_std.csv', index=False)
ci_df.to_csv(REPORT_DIR / 'summary_ci95.csv', index=False)

display(summary)
display(ci_df)
save_progress('summary_saved')


In [ ]:
# McNemar tests using first seed predictions as paired comparison
y = all_true[0]
p_fp = all_preds['fp32'][0]
p_i8 = all_preds['int8'][0]

def mcnemar_p(y_true, p_a, p_b):
    a_ok = (p_a == y_true)
    b_ok = (p_b == y_true)
    b = int(np.sum(a_ok & (~b_ok)))
    c = int(np.sum((~a_ok) & b_ok))
    table = [[0, b], [c, 0]]
    return float(mcnemar(table, exact=False, correction=True).pvalue), b, c

p_fp_i8, b1, c1 = mcnemar_p(y, p_fp, p_i8)
sig = [{'comparison': 'fp32_vs_int8', 'p_value': p_fp_i8, 'b': b1, 'c': c1}]

if len(all_preds['distill']) > 0:
    p_di = all_preds['distill'][0]
    p_fp_di, b2, c2 = mcnemar_p(y, p_fp, p_di)
    sig.append({'comparison': 'fp32_vs_distilled', 'p_value': p_fp_di, 'b': b2, 'c': c2})

sig_df = pd.DataFrame(sig)
sig_df.to_csv(REPORT_DIR / 'significance_mcnemar.csv', index=False)
display(sig_df)
save_progress('significance_saved')


In [ ]:
# Export ONNX of the last evaluated FP32 model
onnx_fp32 = EXPORT_DIR / 'liveness_fp32.onnx'
dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE)
fp32_model = fp32_model.cpu().eval()
torch.onnx.export(fp32_model, dummy, str(onnx_fp32), input_names=['input'], output_names=['logits'], opset_version=13)

summary_best = rdf.groupby('variant', as_index=False)['acer'].mean().sort_values('acer')

matrix_cols = ['timestamp_utc','task','run_id','variant','dataset_train','dataset_eval','model_name','epochs','batch_size','img_size','seed','accuracy','f1','auc','eer','apcer','bpcer','acer','latency_ms','model_size_mb','artifact_path','notes']
if MATRIX_CSV.exists():
    mdf = pd.read_csv(MATRIX_CSV)
else:
    mdf = pd.DataFrame(columns=matrix_cols)

for _, r in summary_best.iterrows():
    vr = rdf[rdf['variant']==r['variant']]
    mdf = pd.concat([mdf, pd.DataFrame([{
      'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
      'task':'liveness', 'run_id':RUN_ID, 'variant':r['variant'],
      'dataset_train':'celeba_spoof_train', 'dataset_eval':'celeba_spoof_test',
      'model_name':'mobilenet_family', 'epochs': EPOCHS_FP32 if r['variant']=='fp32' else EPOCHS_DISTILL,
      'batch_size':BATCH_SIZE, 'img_size':IMG_SIZE, 'seed':'multi',
      'accuracy': vr['acc'].mean(), 'auc': vr['auc'].mean(),
      'apcer': vr['apcer'].mean(), 'bpcer': vr['bpcer'].mean(), 'acer': vr['acer'].mean(),
      'latency_ms': vr['latency_ms'].mean(), 'model_size_mb': vr['size_mb'].mean(),
      'artifact_path': str(RUN_DIR), 'notes':'mean over seeds'
    }])], ignore_index=True)

mdf.to_csv(MATRIX_CSV, index=False)

plt.figure(figsize=(7,4))
sns.barplot(data=rdf, x='variant', y='acer', estimator=np.mean, errorbar='sd')
plt.title('ACER by Variant (mean ± sd over seeds)')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'acer_variant_bar.png', dpi=140)

save_progress('complete', onnx_fp32=str(onnx_fp32), matrix=str(MATRIX_CSV))
print('Complete. ONNX:', onnx_fp32)
